# Genome Preprocessing and Dataset Construction

This notebook prepares the validated *Staphylococcus aureus* genome sequences
for sequence-based ciprofloxacin resistance prediction.

## Workflow

1. Load the genome retrieval manifest.
2. Perform final genome-level data quality checks.
3. Create genome-level train, validation, and test splits.
4. Prevent isolate-level leakage between splits.
5. Optionally evaluate genome relatedness using Mash/MinHash.
6. Generate fixed-length sequence windows independently within each split.
7. Track genomic coordinates and sequence quality for every window.
8. Convert sequence windows into one-hot encoded representations.
9. Prepare the resulting data for CNN training.

## Important Data-Splitting Principle

Genome-level splitting is performed **before** window generation.

Windows from the same genome must never appear across different dataset splits.
This prevents information leakage between training, validation, and test sets.

## Step 1 — Load Final Genome Manifest

The genome retrieval manifest links each phenotype-labeled genome to its
BV-BRC availability and local FASTA sequence.

In [1]:
import pandas as pd
from pathlib import Path

# Load final genome retrieval manifest

manifest_path = next(
    Path("/kaggle/input").rglob("genome_retrieval_manifest.csv")
)

manifest_df = pd.read_csv(
    manifest_path,
    dtype={"Genome ID": str}
)

print("Manifest shape:", manifest_df.shape)

print("\nColumns:")
print(manifest_df.columns.tolist())

print("\nFASTA availability:")
print(
    manifest_df["FASTA Available"].value_counts()
)

Manifest shape: (1205, 5)

Columns:
['Genome ID', 'Phenotype', 'Label', 'Metadata Available', 'FASTA Available']

FASTA availability:
FASTA Available
True     1138
False      67
Name: count, dtype: int64


In [2]:
from pathlib import Path
import torch

# 1. Check GPU availability
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 2. List attached datasets
print("\nAttached datasets:")
input_dir = Path("/kaggle/input")

for path in input_dir.iterdir():
    print(path)

# 3. Locate the genome, processed, and raw data folders automatically
print("\nSearching for data folders:")

genome_dirs = list(input_dir.rglob("genomes"))
processed_dirs = list(input_dir.rglob("processed"))
raw_dirs = list(input_dir.rglob("raw"))

for name, folders in [
    ("genomes", genome_dirs),
    ("processed", processed_dirs),
    ("raw", raw_dirs),
]:
    print(f"\n{name} folders:")
    for folder in folders:
        if folder.is_dir():
            print(folder)

# 4. Verify genome files
if genome_dirs and genome_dirs[0].is_dir():
    genome_dir = genome_dirs[0]
    fasta_files = list(genome_dir.glob("*.fna"))

    print("\nGenome folder exists:", genome_dir.exists())
    print("Genome folder path:", genome_dir)
    print("FASTA files found:", len(fasta_files))
else:
    print("\nGenome folder was not found.")

GPU available: True
GPU: Tesla T4

Attached datasets:
/kaggle/input/datasets

Searching for data folders:

genomes folders:
/kaggle/input/datasets/prajaktananda1905/amr-data/data/genomes

processed folders:
/kaggle/input/datasets/prajaktananda1905/amr-data/data/processed

raw folders:
/kaggle/input/datasets/prajaktananda1905/amr-data/data/raw

Genome folder exists: True
Genome folder path: /kaggle/input/datasets/prajaktananda1905/amr-data/data/genomes
FASTA files found: 1138


In [3]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("*"):
    if path.is_dir() and path.name in {"genomes", "processed", "raw"}:
        print(path)

/kaggle/input/datasets/prajaktananda1905/amr-data/data/genomes
/kaggle/input/datasets/prajaktananda1905/amr-data/data/processed
/kaggle/input/datasets/prajaktananda1905/amr-data/data/raw


## Step 2 — Define the Modeling Cohort

The modeling cohort contains only genomes with successfully retrieved FASTA
sequences.

The 67 genomes unavailable from BV-BRC remain documented in the retrieval
manifest but are excluded from sequence-based modeling.

In [4]:
# Select genomes with successfully retrieved FASTA files

model_df = manifest_df[
    manifest_df["FASTA Available"] == True
].copy()

print("Modeling genomes:", len(model_df))

print("\nLabel distribution:")
print(model_df["Label"].value_counts())

print("\nPhenotype distribution:")
print(model_df["Phenotype"].value_counts())

print("\nUnique Genome IDs:", model_df["Genome ID"].nunique())

Modeling genomes: 1138

Label distribution:
Label
0    664
1    474
Name: count, dtype: int64

Phenotype distribution:
Phenotype
('Susceptible',)    664
('Resistant',)      474
Name: count, dtype: int64

Unique Genome IDs: 1138


## Step 3 — Clean Phenotype Labels

The phenotype column is normalized to plain string labels so that the dataset
contains the intended categorical values:

- Susceptible
- Resistant

The binary `Label` column remains unchanged:
- 0 = Susceptible
- 1 = Resistant

In [5]:
# Normalize phenotype labels

model_df["Phenotype"] = (
    model_df["Phenotype"]
    .astype(str)
    .str.strip("()', ")
)

print("Phenotype distribution:")
print(model_df["Phenotype"].value_counts())

print("\nUnique phenotype values:")
print(model_df["Phenotype"].unique())

Phenotype distribution:
Phenotype
Susceptible    664
Resistant      474
Name: count, dtype: int64

Unique phenotype values:
['Susceptible' 'Resistant']


## Step 4 — Genome-Level Train/Validation/Test Split

The modeling cohort is divided at the genome level using a stratified split:

- 70% training
- 15% validation
- 15% testing

Stratification preserves the Resistant/Susceptible proportion across splits.

The split is performed before genome window generation to prevent windows
originating from the same genome from appearing across different datasets.

In [6]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["Label"],
    random_state=42
)

# Second split: divide the temporary 30% equally
# → 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Label"],
    random_state=42
)

print("Train genomes:", len(train_df))
print("Validation genomes:", len(val_df))
print("Test genomes:", len(test_df))

print("\nTrain label distribution:")
print(train_df["Label"].value_counts())

print("\nValidation label distribution:")
print(val_df["Label"].value_counts())

print("\nTest label distribution:")
print(test_df["Label"].value_counts())

Train genomes: 796
Validation genomes: 171
Test genomes: 171

Train label distribution:
Label
0    464
1    332
Name: count, dtype: int64

Validation label distribution:
Label
0    100
1     71
Name: count, dtype: int64

Test label distribution:
Label
0    100
1     71
Name: count, dtype: int64


## Step 5 — Verify Genome-Level Split Integrity

The train, validation, and test sets must contain completely disjoint Genome IDs.

This prevents genomic windows originating from the same isolate from appearing
in multiple dataset partitions and causing data leakage.

In [7]:
# Verify that Genome IDs are completely disjoint

train_ids = set(train_df["Genome ID"])
val_ids = set(val_df["Genome ID"])
test_ids = set(test_df["Genome ID"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(val_test_overlap))

print("\nTotal unique Genome IDs across all splits:",
      len(train_ids | val_ids | test_ids))

print("Expected unique Genome IDs:", len(model_df))

if not train_val_overlap and not train_test_overlap and not val_test_overlap:
    print("\n✓ No genome-level leakage detected.")
else:
    print("\n✗ Genome overlap detected.")

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Total unique Genome IDs across all splits: 1138
Expected unique Genome IDs: 1138

✓ No genome-level leakage detected.


## Step 6 — Define Genome Windowing

Each genome is processed contig-by-contig rather than concatenated into a
single artificial sequence.

Each contig is divided into overlapping windows using:

- Window size: 500 bp
- Stride: 250 bp

For every generated window, we retain:

- Genome ID
- Contig ID
- Start coordinate
- End coordinate
- Strand
- Sequence
- Fraction of ambiguous bases (N)

Windows are generated only after the genome-level train/validation/test split,
ensuring that windows from the same isolate cannot cross dataset boundaries.

In [8]:
# Windowing configuration

WINDOW_SIZE = 500
STRIDE = 250

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)

Window size: 500
Stride: 250


## Step 7 — Parse Genome FASTA into Contigs

Genome FASTA files may contain multiple contigs. Each contig is treated as an
independent nucleotide sequence.

We parse the FASTA file into a dictionary mapping each contig identifier to
its nucleotide sequence.

Contigs are never concatenated across boundaries.

In [9]:
from pathlib import Path

def parse_fasta(fasta_path):
    """
    Parse a FASTA file into a dictionary:
    {contig_id: nucleotide_sequence}
    """
    contigs = {}
    current_id = None
    sequence_parts = []

    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if current_id is not None:
                    contigs[current_id] = "".join(sequence_parts)

                current_id = line[1:].split()[0]
                sequence_parts = []

            else:
                sequence_parts.append(line.upper())

        if current_id is not None:
            contigs[current_id] = "".join(sequence_parts)

    return contigs

In [10]:
from pathlib import Path

# Automatically locate the genome folder in the attached Kaggle dataset
input_dir = Path("/kaggle/input")
genome_matches = list(input_dir.rglob("genomes"))

genome_dirs = [
    path for path in genome_matches
    if path.is_dir() and path.parent.name == "data"
]

if not genome_dirs:
    raise FileNotFoundError("Could not find the genomes folder in Kaggle input.")

genome_dir = genome_dirs[0]

print("Genome directory:", genome_dir)
print("Directory exists:", genome_dir.exists())
print("FASTA files found:", len(list(genome_dir.glob("*.fna"))))

Genome directory: /kaggle/input/datasets/prajaktananda1905/amr-data/data/genomes
Directory exists: True
FASTA files found: 1138


In [11]:
# Test FASTA parsing on the representative genome

test_genome_id = "1280.15865"
test_fasta_path = genome_dir / f"{test_genome_id}.fna"

test_contigs = parse_fasta(test_fasta_path)

print("Genome ID:", test_genome_id)
print("Number of contigs:", len(test_contigs))

print("\nFirst 5 contigs:")
for contig_id, sequence in list(test_contigs.items())[:5]:
    print(contig_id, "—", len(sequence), "bp")

Genome ID: 1280.15865
Number of contigs: 26

First 5 contigs:
accn|LKNK01000018 — 87177 bp
accn|LKNK01000003 — 198837 bp
accn|LKNK01000016 — 83819 bp
accn|LKNK01000010 — 130579 bp
accn|LKNK01000006 — 24070 bp


## Step 8 — Define Genome Windowing Function

Each genome is processed contig-by-contig.

For each contig, overlapping windows are generated using:
- Window size: 500 bp
- Stride: 250 bp

Windows are never generated across contig boundaries.

Each window retains:
- Genome ID
- Contig ID
- Start coordinate
- End coordinate
- Strand
- Sequence
- Fraction of ambiguous bases (N)

Window generation occurs only after genome-level dataset splitting,
preventing windows from the same isolate from appearing across different
dataset partitions.

In [12]:
# Windowing configuration

WINDOW_SIZE = 500
STRIDE = 250

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)

Window size: 500
Stride: 250


In [13]:
def generate_windows(sequence, window_size=500, stride=250):
    """
    Generate overlapping windows from a single contig sequence.

    Windows are generated only within the contig.
    The final incomplete window is discarded.
    """
    windows = []

    for start in range(0, len(sequence) - window_size + 1, stride):
        end = start + window_size
        window_sequence = sequence[start:end]

        n_fraction = window_sequence.upper().count("N") / window_size

        windows.append({
            "Start": start,
            "End": end,
            "Sequence": window_sequence,
            "N_fraction": n_fraction
        })

    return windows

In [14]:
# Test window generation on one contig

first_contig_id, first_contig_sequence = next(iter(test_contigs.items()))

test_windows = generate_windows(
    first_contig_sequence,
    WINDOW_SIZE,
    STRIDE
)

print("Contig ID:", first_contig_id)
print("Contig length:", len(first_contig_sequence), "bp")
print("Number of windows:", len(test_windows))

print("\nFirst window:")
print(test_windows[0])

Contig ID: accn|LKNK01000018
Contig length: 87177 bp
Number of windows: 347

First window:
{'Start': 0, 'End': 500, 'Sequence': 'TCTTACTGCTGTTTTTTTAGGGATTTATGTCCCAGCCTCTTTTTGGTTTTAATTTTACAATAATTGCTTAGTATCTTTTAAATCAAATGTGCAATTGTTAATATAGCTCTATATTAGAGTGTTTTGGATGAAACATTTGCTATGATAAGGTAAAATTAAATTAAGAAAGTGCTTTAATGCTTTAACTGTATAAATCATTTAAATAGTTTTGCTACAACGAATATGTTATTATAAGTTACCAAAATATTTTAGAGAAAGAAATCAAACAAATAAATTGTATTTTGTTTGATGCTTTAATACAATGTGGATACTATTCATCGACTGTAAGGAGTGAGATGATATGAGAATAGAACGAGTAGATGATACAACTGTAAAATTGTTTATAACATATAGCGATATCGAGGCCCGTGGATTTAGTCGTGAAGATTTATGGACAAATCGCAAACGTGGCGAAGAATTCTTTTGGTCAATGATGGATGAAATTAACGAAGAAGAAGATT', 'N_fraction': 0.0}


In [15]:
# Inspect the first few window coordinates

for window in test_windows[:5]:
    print(
        "Start:", window["Start"],
        "| End:", window["End"],
        "| Length:", len(window["Sequence"]),
        "| N fraction:", window["N_fraction"]
    )

Start: 0 | End: 500 | Length: 500 | N fraction: 0.0
Start: 250 | End: 750 | Length: 500 | N fraction: 0.0
Start: 500 | End: 1000 | Length: 500 | N fraction: 0.0
Start: 750 | End: 1250 | Length: 500 | N fraction: 0.0
Start: 1000 | End: 1500 | Length: 500 | N fraction: 0.0


In [17]:
# Test windowing on several contigs

# Test windowing on several contigs

test_contigs_sample = list(test_contigs)[:5]

for i, item in enumerate(test_contigs_sample):
    # Support either (contig_id, sequence) tuples or plain sequences
    if isinstance(item, tuple) and len(item) == 2:
        contig_id, sequence = item
    else:
        contig_id = f"contig_{i + 1}"
        sequence = item

    windows = generate_windows(
        sequence,
        window_size=WINDOW_SIZE,
        stride=STRIDE
    )

    print(
        contig_id,
        "| Contig length:", len(sequence),
        "| Windows:", len(windows)
    )

contig_1 | Contig length: 17 | Windows: 0
contig_2 | Contig length: 17 | Windows: 0
contig_3 | Contig length: 17 | Windows: 0
contig_4 | Contig length: 17 | Windows: 0
contig_5 | Contig length: 17 | Windows: 0


## Step 12 — Estimate Total Genome Windows

Before generating the complete window-level dataset, we first estimate the
number of windows that will be produced from the modeling cohort.

This provides an approximate dataset size and helps determine whether the
window-level data can be safely constructed in memory or should be processed
in batches.

For each genome:

1. Load the corresponding FASTA file.
2. Parse the genome into individual contigs.
3. Generate windows independently within each contig.
4. Count the resulting windows.
5. Sum the counts across all genomes in the modeling cohort.

The current windowing configuration is:

- Window size: **500 bp**
- Stride: **250 bp**
- Contigs are processed independently.
- Windows are generated only from genomes belonging to the modeling cohort.

No window-level data is saved at this stage; this step only estimates the
total number of windows that will be generated.

In [18]:
# Estimate total number of windows in the modeling cohort

total_windows = 0
genomes_with_files = 0

for genome_id in model_df["Genome ID"]:
    fasta_path = genome_dir / f"{genome_id}.fna"

    if not fasta_path.exists():
        continue

    contigs = parse_fasta(fasta_path)

    for sequence in contigs.values():
        total_windows += len(
            generate_windows(
                sequence,
                window_size=WINDOW_SIZE,
                stride=STRIDE
            )
        )

    genomes_with_files += 1

print("Genomes available:", genomes_with_files)
print("Estimated total windows:", total_windows)

Genomes available: 1138
Estimated total windows: 12701599


## Step 13 — Estimate Windows Per Dataset Split

The modeling cohort has already been divided at the genome level into
training, validation, and test sets.

Before constructing the complete window-level dataset, we estimate the
number of windows produced by each split.

This step is important because the number of windows can be very large,
especially when using overlapping 500 bp windows with a 250 bp stride.

The estimate will guide the subsequent dataset-construction strategy and
help avoid loading the complete window-level dataset into memory at once.

Windows remain associated with their source genome and are generated only
after the genome-level split, preserving the separation between training,
validation, and test genomes.

In [19]:
# Estimate the number of windows in each dataset split

def estimate_split_windows(split_df):
    total_windows = 0
    genomes_processed = 0

    for genome_id in split_df["Genome ID"]:
        fasta_path = genome_dir / f"{genome_id}.fna"

        if not fasta_path.exists():
            continue

        contigs = parse_fasta(fasta_path)

        for sequence in contigs.values():
            total_windows += len(
                generate_windows(
                    sequence,
                    window_size=WINDOW_SIZE,
                    stride=STRIDE
                )
            )

        genomes_processed += 1

    return genomes_processed, total_windows


train_genomes, train_windows = estimate_split_windows(train_df)
val_genomes, val_windows = estimate_split_windows(val_df)
test_genomes, test_windows = estimate_split_windows(test_df)

print("Training genomes:", train_genomes)
print("Training windows:", train_windows)

print("\nValidation genomes:", val_genomes)
print("Validation windows:", val_windows)

print("\nTest genomes:", test_genomes)
print("Test windows:", test_windows)

print("\nTotal windows:", train_windows + val_windows + test_windows)

Training genomes: 796
Training windows: 8878455

Validation genomes: 171
Validation windows: 1912079

Test genomes: 171
Test windows: 1911065

Total windows: 12701599


## Step 14 — Define Genome-Level Window Processing

Each genome is treated as one multiple-instance learning (MIL) bag.

The genome is first parsed into its individual contigs, and windows are
generated independently within each contig using the configured window size
and stride.

Each generated window retains its genomic provenance:

- Genome ID
- Phenotype
- Binary label
- Contig ID
- Start coordinate
- End coordinate
- Strand
- N fraction
- Sequence

The windows are therefore treated as instances belonging to a particular
genome rather than as independent training examples.

This distinction is essential because the final prediction target is the
genome-level antimicrobial-resistance phenotype.

In [26]:
def process_genome(genome_id):
    """
    Process one genome into its window-level MIL instances.

    Each genome remains a single MIL bag, while individual windows
    retain their genomic coordinates and source information.
    """

    # Locate FASTA file
    fasta_path = genome_dir / f"{genome_id}.fna"

    if not fasta_path.exists():
        raise FileNotFoundError(
            f"FASTA file not found for Genome ID: {genome_id}"
        )

    # Get genome-level label information
    genome_row = model_df[
        model_df["Genome ID"] == genome_id
    ]

    if len(genome_row) != 1:
        raise ValueError(
            f"Expected exactly one phenotype record for {genome_id}, "
            f"found {len(genome_row)}"
        )

    phenotype = genome_row.iloc[0]["Phenotype"]
    label = int(genome_row.iloc[0]["Label"])

    # Parse genome into contigs
    contigs = parse_fasta(fasta_path)

    windows = []

    # Generate windows independently for each contig
    for contig_id, sequence in contigs.items():

        contig_windows = generate_windows(
            sequence,
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        for window in contig_windows:

            windows.append({
                "Genome ID": genome_id,
                "Phenotype": phenotype,
                "Label": label,
                "Contig ID": contig_id,
                "Start": window["Start"],
                "End": window["End"],
                "Strand": "+",
                "Sequence": window["Sequence"],
                "N_fraction": window["N_fraction"]
            })

    return windows

In [27]:
test_genome_id = "1280.15865"

test_genome_windows = process_genome(test_genome_id)

print("Genome ID:", test_genome_id)
print("Total windows:", len(test_genome_windows))

print("\nFirst window:")
print(test_genome_windows[0])

print("\nLast window:")
print(test_genome_windows[-1])

Genome ID: 1280.15865
Total windows: 10885

First window:
{'Genome ID': '1280.15865', 'Phenotype': 'Susceptible', 'Label': 0, 'Contig ID': 'accn|LKNK01000018', 'Start': 0, 'End': 500, 'Strand': '+', 'Sequence': 'TCTTACTGCTGTTTTTTTAGGGATTTATGTCCCAGCCTCTTTTTGGTTTTAATTTTACAATAATTGCTTAGTATCTTTTAAATCAAATGTGCAATTGTTAATATAGCTCTATATTAGAGTGTTTTGGATGAAACATTTGCTATGATAAGGTAAAATTAAATTAAGAAAGTGCTTTAATGCTTTAACTGTATAAATCATTTAAATAGTTTTGCTACAACGAATATGTTATTATAAGTTACCAAAATATTTTAGAGAAAGAAATCAAACAAATAAATTGTATTTTGTTTGATGCTTTAATACAATGTGGATACTATTCATCGACTGTAAGGAGTGAGATGATATGAGAATAGAACGAGTAGATGATACAACTGTAAAATTGTTTATAACATATAGCGATATCGAGGCCCGTGGATTTAGTCGTGAAGATTTATGGACAAATCGCAAACGTGGCGAAGAATTCTTTTGGTCAATGATGGATGAAATTAACGAAGAAGAAGATT', 'N_fraction': 0.0}

Last window:
{'Genome ID': '1280.15865', 'Phenotype': 'Susceptible', 'Label': 0, 'Contig ID': 'accn|LKNK01000023', 'Start': 17750, 'End': 18250, 'Strand': '+', 'Sequence': 'GTCATGATTGGTAGAGGCATTTTCCATAATCCATTCGCTTTTGAAAAAGAACCACGCGAACACACAAGCAAAGAACTATTAGATTTGTTGAG

## Step 16 — Validate Window Integrity

The generated windows are validated before being used for model training.

The validation checks:

- Every window has the expected length of 500 bp.
- Window coordinates correspond to the sequence length.
- Only valid nucleotide characters (`A`, `C`, `G`, `T`, `N`) are present.
- The reported `N_fraction` agrees with the number of ambiguous bases.
- Windows do not extend beyond their source contig boundaries.

These checks ensure that the window-generation process produces valid,
well-defined genomic instances before numerical sequence representation and
model training.

In [28]:
# Validate window integrity for the test genome

valid_bases = set("ACGTN")

invalid_length = []
invalid_coordinates = []
invalid_bases = []
invalid_n_fraction = []

for window in test_genome_windows:

    sequence = window["Sequence"]
    start = window["Start"]
    end = window["End"]

    # Check sequence length
    if len(sequence) != WINDOW_SIZE:
        invalid_length.append(window)

    # Check coordinate length
    if end - start != WINDOW_SIZE:
        invalid_coordinates.append(window)

    # Check nucleotide characters
    if not set(sequence).issubset(valid_bases):
        invalid_bases.append(window)

    # Check N fraction
    expected_n_fraction = sequence.count("N") / len(sequence)

    if abs(window["N_fraction"] - expected_n_fraction) > 1e-9:
        invalid_n_fraction.append(window)


print("Total windows:", len(test_genome_windows))

print("Invalid sequence lengths:", len(invalid_length))
print("Invalid coordinates:", len(invalid_coordinates))
print("Invalid nucleotide characters:", len(invalid_bases))
print("Incorrect N fractions:", len(invalid_n_fraction))

if not any([
    invalid_length,
    invalid_coordinates,
    invalid_bases,
    invalid_n_fraction
]):
    print("\n✓ Window integrity validation passed.")
else:
    print("\n✗ Window integrity problems detected.")

Total windows: 10885
Invalid sequence lengths: 0
Invalid coordinates: 0
Invalid nucleotide characters: 0
Incorrect N fractions: 0

✓ Window integrity validation passed.


## Step 17 — Validate Window Coverage

The windowing procedure is further validated by comparing the genomic
sequence covered by the generated windows with the source contig lengths.

Because the windows use a 500 bp size and 250 bp stride, overlapping windows
are expected. Therefore, this check verifies the number and coordinate range
of windows rather than requiring every base to occur only once.

This confirms that windows remain within their respective contigs and that
the window-generation procedure is consistent across contigs.

In [30]:

# Validate window coverage for the test genome

coverage_errors = []

# Normalize test_contigs into (contig_id, sequence) pairs
if isinstance(test_contigs, dict):
    contig_items = list(test_contigs.items())
else:
    contig_items = []

    for i, item in enumerate(test_contigs):
        if isinstance(item, dict):
            contig_id = item.get("Contig ID", item.get("contig_id", str(i)))
            sequence = item.get("Sequence", item.get("sequence"))

        elif isinstance(item, (tuple, list)):
            # The sequence is expected to be the final item.
            if len(item) < 2:
                coverage_errors.append((str(i), "Invalid contig record"))
                continue

            contig_id = item[0]
            sequence = item[-1]

        else:
            # A plain sequence string
            contig_id = str(i)
            sequence = item

        if sequence is None:
            coverage_errors.append((str(contig_id), "Missing sequence"))
            continue

        contig_items.append((contig_id, sequence))

for contig_id, sequence in contig_items:
    contig_length = len(sequence)

    contig_windows = [
        window for window in test_genome_windows
        if window["Contig ID"] == contig_id
    ]

    if not contig_windows:
        coverage_errors.append((contig_id, "No windows generated"))
        continue

    # Sort windows by start position before checking coverage
    contig_windows.sort(key=lambda window: window["Start"])

    if contig_windows[0]["Start"] != 0:
        coverage_errors.append(
            (contig_id, "First window does not start at 0")
        )

    for window in contig_windows:
        if window["End"] > contig_length:
            coverage_errors.append(
                (contig_id, "Window extends beyond contig")
            )
            break

    if contig_windows[-1]["End"] > contig_length:
        coverage_errors.append(
            (contig_id, "Last window exceeds contig length")
        )

print("Total contigs:", len(contig_items))
print("Contigs with coverage errors:", len(coverage_errors))

if not coverage_errors:
    print("\nWindow coverage validation passed.")
else:
    print("\nCoverage problems detected.")
    print(coverage_errors[:10])


Total contigs: 26
Contigs with coverage errors: 0

Window coverage validation passed.


## Step 18 — Define One-Hot Sequence Encoding

Each 500 bp DNA window is converted into a numerical representation
suitable for the CNN.

Each nucleotide is represented using a four-dimensional one-hot vector:

- A → [1, 0, 0, 0]
- C → [0, 1, 0, 0]
- G → [0, 0, 1, 0]
- T → [0, 0, 0, 1]

Ambiguous nucleotide `N` is represented as:

- N → [0, 0, 0, 0]

The resulting representation for a 500 bp window has shape:

    (500, 4)

This representation preserves the nucleotide sequence while allowing the
CNN to learn local sequence patterns.

In [31]:
# Define one-hot encoding for DNA sequences

import numpy as np

BASE_TO_VECTOR = {
    "A": [1, 0, 0, 0],
    "C": [0, 1, 0, 0],
    "G": [0, 0, 1, 0],
    "T": [0, 0, 0, 1],
    "N": [0, 0, 0, 0]
}


def one_hot_encode(sequence):
    """
    Convert a DNA sequence into a one-hot encoded NumPy array.

    Output shape:
        (sequence_length, 4)
    """

    sequence = sequence.upper()

    encoded = np.array(
        [BASE_TO_VECTOR[base] for base in sequence],
        dtype=np.float32
    )

    return encoded

## Step 19 — Test One-Hot Encoding

A representative 500 bp window is encoded to verify that the one-hot
representation has the expected dimensions and nucleotide mappings.

The test checks:

- Output shape is `(500, 4)`
- Each standard nucleotide maps to exactly one active channel
- Ambiguous `N` bases are encoded as `[0, 0, 0, 0]`

In [33]:
# Test one-hot encoding on the first genome window

test_sequence = test_genome_windows[0]["Sequence"]

encoded_test = one_hot_encode(test_sequence)

print("Original sequence length:", len(test_sequence))
print("Encoded shape:", encoded_test.shape)

print("\nFirst 10 bases:")
print(test_sequence[:10])

print("\nFirst 10 encoded rows:")
print(encoded_test[:10])

print("\nUnique encoded vectors:")
print(np.unique(encoded_test, axis=0))

if encoded_test.shape == (500, 4):
    print("\n✓ One-hot encoding shape is correct.")
else:
    print("\n✗ Unexpected encoding shape.")

Original sequence length: 500
Encoded shape: (500, 4)

First 10 bases:
TCTTACTGCT

First 10 encoded rows:
[[0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]

Unique encoded vectors:
[[0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]

✓ One-hot encoding shape is correct.


## Step 20 — Validate Ambiguous Base Encoding

Ambiguous nucleotide bases are represented by `N`.

To avoid assigning an arbitrary nucleotide identity to an ambiguous base,
`N` is encoded as:

    N → [0, 0, 0, 0]

This test explicitly verifies that the encoder handles ambiguous bases
according to the defined representation.

In [34]:
# Test ambiguous N-base encoding

n_test_sequence = "ACGTN"

n_encoded = one_hot_encode(n_test_sequence)

print("Test sequence:", n_test_sequence)
print("\nEncoded representation:")
print(n_encoded)

print("\nEncoding of N:")
print(n_encoded[4])

if np.array_equal(n_encoded[4], [0, 0, 0, 0]):
    print("\n✓ N is correctly encoded as [0, 0, 0, 0].")
else:
    print("\n✗ N encoding is incorrect.")

Test sequence: ACGTN

Encoded representation:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]]

Encoding of N:
[0. 0. 0. 0.]

✓ N is correctly encoded as [0, 0, 0, 0].


## Step 21 — Validate Encoding Across the Test Genome

The one-hot encoder is tested across all windows generated from the
representative genome.

This verifies that:

- Every window produces the expected `(500, 4)` representation.
- All encoded values are valid binary values.
- No unexpected nucleotide encoding occurs.

The complete dataset will not be materialized in memory at this stage.
This test is restricted to one genome before scaling to the full cohort.

In [35]:
# Validate one-hot encoding across all windows of the test genome

encoded_windows = []

for window in test_genome_windows:
    encoded = one_hot_encode(window["Sequence"])
    encoded_windows.append(encoded)

# Convert to a single test array
encoded_windows_array = np.stack(encoded_windows)

print("Number of windows:", len(encoded_windows_array))
print("Encoded array shape:", encoded_windows_array.shape)

print("Minimum value:", encoded_windows_array.min())
print("Maximum value:", encoded_windows_array.max())

# Validate dimensions and values
shape_valid = encoded_windows_array.shape == (len(test_genome_windows), 500, 4)
values_valid = np.all(
    np.isin(encoded_windows_array, [0.0, 1.0])
)

if shape_valid and values_valid:
    print("\n✓ Genome-wide one-hot encoding validation passed.")
else:
    print("\n✗ Encoding validation failed.")

Number of windows: 10885
Encoded array shape: (10885, 500, 4)
Minimum value: 0.0
Maximum value: 1.0

✓ Genome-wide one-hot encoding validation passed.


## Step 22 — Define Streaming Window Batches

The complete genome cohort contains millions of overlapping windows.
Therefore, all windows will not be one-hot encoded and stored simultaneously.

Instead, windows from each genome are processed in manageable batches.

For each genome:

1. Generate its windows.
2. Divide the windows into batches.
3. One-hot encode each batch.
4. Process the batch through the CNN.
5. Retain the resulting window embeddings for genome-level aggregation.

This preserves the multiple-instance learning structure while controlling
memory usage during model training.

In [36]:
# Window batch size
WINDOW_BATCH_SIZE = 128


def generate_window_batches(windows, batch_size=WINDOW_BATCH_SIZE):
    """
    Yield one-hot encoded batches of genome windows.

    Each batch has shape:
        (batch_size, 500, 4)

    The final batch may contain fewer than batch_size windows.
    """
    
    for start in range(0, len(windows), batch_size):
        batch_windows = windows[start:start + batch_size]

        batch_encoded = np.stack([
            one_hot_encode(window["Sequence"])
            for window in batch_windows
        ])

        yield batch_encoded

In [37]:
print("Type of test_windows:", type(test_windows))
print("Value of test_windows:", test_windows if isinstance(test_windows, int) else "not an integer")

print("\nType of test_genome_windows:", type(test_genome_windows))
print("Type of test_windows:", type(test_windows))

Type of test_windows: <class 'int'>
Value of test_windows: 1911065

Type of test_genome_windows: <class 'list'>
Type of test_windows: <class 'int'>


In [38]:
# Use the actual window list from the representative genome
representative_windows = test_genome_windows

print("Type:", type(representative_windows))
print("Number of windows:", len(representative_windows))

Type: <class 'list'>
Number of windows: 10885


In [39]:
# Test streaming batches on the representative genome

batch_generator = generate_window_batches(
    representative_windows,
    batch_size=WINDOW_BATCH_SIZE
)

batch_count = 0
total_windows = 0

for batch in batch_generator:
    batch_count += 1
    total_windows += batch.shape[0]

    if batch_count <= 3:
        print(f"Batch {batch_count}:", batch.shape)

print("\nTotal batches:", batch_count)
print("Total windows processed:", total_windows)
print("Expected windows:", len(representative_windows))

if total_windows == len(representative_windows):
    print("\n✓ All windows were processed exactly once.")
else:
    print("\n✗ Window count mismatch.")

Batch 1: (128, 500, 4)
Batch 2: (128, 500, 4)
Batch 3: (128, 500, 4)

Total batches: 86
Total windows processed: 10885
Expected windows: 10885

✓ All windows were processed exactly once.


## Step 24 — Define the CNN Feature Extractor

The CNN operates on individual 500-bp genomic windows represented using
one-hot encoding.

Its purpose is to learn local sequence features such as nucleotide
patterns and motifs that may contribute to antimicrobial resistance.

The CNN does not directly produce the final genome-level prediction.

Instead, it transforms each genomic window into a lower-dimensional
feature representation (window embedding).

These window embeddings are subsequently aggregated using attention-based
multiple-instance learning to produce a genome-level prediction.

In [40]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


## Step 23 — CNN Feature Extraction

A convolutional neural network (CNN) is used to learn local sequence patterns
from one-hot encoded 500-bp genomic windows.

Each window is represented as:

    (500, 4)

where the four channels correspond to the nucleotide bases A, C, G, and T.

PyTorch Conv1D layers expect the input in the format:

    (batch_size, channels, sequence_length)

Therefore, each encoded window is rearranged from:

    (batch_size, 500, 4)

to:

    (batch_size, 4, 500)

The CNN processes each genomic window independently and produces a
128-dimensional embedding for every window.

These window-level embeddings are subsequently passed to the attention-based
multiple-instance learning (MIL) component, which aggregates information across
all windows belonging to the same genome.

The CNN is therefore a feature extractor rather than the final genome-level
classifier.

In [41]:
import torch
import torch.nn as nn


class GenomeCNN(nn.Module):
    """
    CNN feature extractor for one-hot encoded DNA windows.

    Input:
        (batch_size, 500, 4)

    Output:
        (batch_size, 128)

    Each 500-bp genomic window is converted into a
    128-dimensional learned embedding.
    """

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=4,
            out_channels=64,
            kernel_size=15,
            padding=7
        )

        self.conv2 = nn.Conv1d(
            in_channels=64,
            out_channels=128,
            kernel_size=15,
            padding=7
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        self.global_pool = nn.AdaptiveMaxPool1d(1)

        self.embedding = nn.Linear(
            128,
            128
        )

    def forward(self, x):
        """
        Forward pass.

        Input:
            x shape = (batch_size, 500, 4)

        Output:
            shape = (batch_size, 128)
        """

        # Rearrange for Conv1D:
        # (batch, 500, 4) → (batch, 4, 500)
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.pool(x)

        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # Global pooling across sequence positions
        x = self.global_pool(x)

        # Remove final sequence dimension
        x = x.squeeze(-1)

        # Produce final window embedding
        x = self.embedding(x)

        return x

In [42]:
# Create the CNN feature extractor

cnn = GenomeCNN()

print(cnn)

GenomeCNN(
  (conv1): Conv1d(4, 64, kernel_size=(15,), stride=(1,), padding=(7,))
  (conv2): Conv1d(64, 128, kernel_size=(15,), stride=(1,), padding=(7,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveMaxPool1d(output_size=1)
  (embedding): Linear(in_features=128, out_features=128, bias=True)
)


In [43]:
# Take the first 128 encoded windows from the representative genome

test_batch = encoded_windows_array[:128]

print("Input shape:", test_batch.shape)

# Convert NumPy array to PyTorch tensor
test_tensor = torch.tensor(
    test_batch,
    dtype=torch.float32
)

# Pass through CNN
with torch.no_grad():
    test_embeddings = cnn(test_tensor)

print("Output shape:", test_embeddings.shape)

Input shape: (128, 500, 4)
Output shape: torch.Size([128, 128])


## Step 25 — Attention-Based Multiple-Instance Learning

The CNN produces a learned embedding for each genomic window.

However, the final prediction is made at the genome level rather than the
window level.

Therefore, all windows belonging to a genome are treated as a single
multiple-instance learning (MIL) bag.

For a genome containing N windows:

    Window 1 → embedding 1
    Window 2 → embedding 2
    ...
    Window N → embedding N

An attention mechanism learns an importance weight for each window embedding.

The attention weights determine which genomic regions contribute more strongly
to the genome-level representation.

The weighted window embeddings are then aggregated into a single genome
embedding, which is used for Resistant/Susceptible classification.

This allows the model to learn which genomic regions are most informative for
ciprofloxacin resistance without assigning an independent resistance label
to every window.

In [45]:
import torch
import torch.nn as nn


class AttentionMIL(nn.Module):
    """
    Attention-based multiple-instance learning aggregator.

    Input:
        (number_of_windows, 128)

    Output:
        genome_embedding: (128,)
        attention_weights: (number_of_windows,)
        prediction_logit: scalar
    """

    def __init__(self, embedding_dim=128):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.classifier = nn.Linear(
            embedding_dim,
            1
        )

    def forward(self, embeddings):
        """
        embeddings:
            Tensor of shape (number_of_windows, embedding_dim)
        """

        # Calculate an attention score for every window
        attention_scores = self.attention(embeddings)

        # Convert scores into normalized attention weights
        attention_weights = torch.softmax(
            attention_scores,
            dim=0
        )

        # Weighted aggregation of all window embeddings
        genome_embedding = torch.sum(
            attention_weights * embeddings,
            dim=0
        )

        # Genome-level prediction
        prediction_logit = self.classifier(
            genome_embedding
        )

        return (
            genome_embedding,
            attention_weights.squeeze(-1),
            prediction_logit.squeeze(-1)
        )

In [46]:
# Create the attention-based MIL aggregator

mil = AttentionMIL(
    embedding_dim=128
)

print(mil)

AttentionMIL(
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (classifier): Linear(in_features=128, out_features=1, bias=True)
)


In [47]:
# Test attention MIL on the CNN embeddings

with torch.no_grad():
    genome_embedding, attention_weights, prediction_logit = mil(
        test_embeddings
    )

print("Window embeddings shape:", test_embeddings.shape)
print("Genome embedding shape:", genome_embedding.shape)
print("Attention weights shape:", attention_weights.shape)
print("Prediction logit shape:", prediction_logit.shape)

print("\nAttention weights sum:", attention_weights.sum().item())

Window embeddings shape: torch.Size([128, 128])
Genome embedding shape: torch.Size([128])
Attention weights shape: torch.Size([128])
Prediction logit shape: torch.Size([])

Attention weights sum: 1.000000238418579


## Step 27 — Combined CNN and Attention-MIL Model

The complete model combines the CNN feature extractor with the
attention-based multiple-instance learning (MIL) aggregator.

For each genome:

1. Genomic windows are generated from the FASTA sequence.
2. Windows are one-hot encoded.
3. Windows are processed through the CNN in memory-efficient batches.
4. The CNN produces a learned embedding for every window.
5. Embeddings from all windows belonging to the genome are combined.
6. Attention MIL assigns an importance weight to each window.
7. The weighted embeddings are aggregated into a single genome-level
   representation.
8. A classifier predicts ciprofloxacin resistance or susceptibility.

The CNN therefore learns local sequence representations, while the
attention-MIL component learns which genomic regions are most informative
for the genome-level AMR phenotype.

In [48]:
class GenomeAMRModel(nn.Module):
    """
    Complete genome-level AMR prediction model.

    CNN:
        Converts each 500-bp DNA window into a 128-dimensional embedding.

    Attention MIL:
        Aggregates all window embeddings belonging to one genome and
        produces a genome-level prediction.
    """

    def __init__(self):
        super().__init__()

        self.cnn = GenomeCNN(
        )

        self.mil = AttentionMIL(
            embedding_dim=128
        )

    def forward(self, window_batch):
        """
        Input:
            window_batch:
                (number_of_windows, 500, 4)

        Output:
            genome_embedding:
                (128,)

            attention_weights:
                (number_of_windows,)

            prediction_logit:
                scalar
        """

        # Extract a learned representation for every window
        window_embeddings = self.cnn(
            window_batch
        )

        # Aggregate window representations at genome level
        genome_embedding, attention_weights, prediction_logit = self.mil(
            window_embeddings
        )

        return (
            genome_embedding,
            attention_weights,
            prediction_logit
        )

In [49]:
# Create the complete CNN + Attention-MIL model

model = GenomeAMRModel()

print(model)

GenomeAMRModel(
  (cnn): GenomeCNN(
    (conv1): Conv1d(4, 64, kernel_size=(15,), stride=(1,), padding=(7,))
    (conv2): Conv1d(64, 128, kernel_size=(15,), stride=(1,), padding=(7,))
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (global_pool): AdaptiveMaxPool1d(output_size=1)
    (embedding): Linear(in_features=128, out_features=128, bias=True)
  )
  (mil): AttentionMIL(
    (attention): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=1, bias=True)
    )
    (classifier): Linear(in_features=128, out_features=1, bias=True)
  )
)


## Step 28 — Test the Combined CNN + Attention-MIL Model

The complete model combines the CNN feature extractor and attention-based
multiple-instance learning into a single genome-level prediction pipeline.

A batch of one-hot encoded genomic windows is first processed by the shared
CNN to obtain window-level embeddings. These embeddings are then aggregated
by the attention-MIL module into a single genome-level representation and
prediction.

This test verifies that the complete forward pass works correctly before
training is implemented.

In [51]:
# Find the window-batching generator currently defined in the notebook

[name for name in globals() if "batch" in name.lower() and callable(globals()[name])]

['generate_window_batches']

In [52]:
# Test the complete CNN + Attention-MIL model

# Generate the first streaming batch from the representative genome
test_batch = next(
    generate_window_batches(
        representative_windows,
        batch_size=128
    )
)

print("Input batch shape:", test_batch.shape)

# Convert NumPy array to PyTorch tensor
test_tensor = torch.tensor(
    test_batch,
    dtype=torch.float32
)

# Run the complete CNN + Attention-MIL model
with torch.no_grad():
    genome_embedding, attention_weights, prediction_logit = model(test_tensor)

print("Genome embedding shape:", genome_embedding.shape)
print("Attention weights shape:", attention_weights.shape)
print("Prediction logit:", prediction_logit)
print("Prediction shape:", prediction_logit.shape)

Input batch shape: (128, 500, 4)
Genome embedding shape: torch.Size([128])
Attention weights shape: torch.Size([128])
Prediction logit: tensor(-0.1620)
Prediction shape: torch.Size([])


## Step 29 — Genome-Level Streaming Inference

The complete genome contains thousands of genomic windows, which cannot be processed
simultaneously without unnecessary memory usage.

The streaming pipeline processes genomic windows in batches:

1. Generate 500-bp windows from the genome.
2. One-hot encode a batch of windows.
3. Pass the batch through the CNN.
4. Collect the resulting 128-dimensional window embeddings.
5. Continue until all genome windows have been processed.
6. Aggregate the complete set of window embeddings using Attention-MIL.
7. Produce one genome-level prediction.

This allows the model to process complete bacterial genomes while keeping
memory usage bounded by the batch size.

In [53]:
# Process the complete representative genome in streaming batches

batch_size = 128

embedding_batches = []
total_windows_processed = 0

with torch.no_grad():

    for batch_number, batch in enumerate(
        generate_window_batches(
            representative_windows,
            batch_size=batch_size
        ),
        start=1
    ):

        # Convert NumPy batch to PyTorch tensor
        batch_tensor = torch.tensor(
            batch,
            dtype=torch.float32
        )

        # Extract a 128-dimensional embedding for each window
        batch_embeddings = model.cnn(batch_tensor)

        # Move embeddings to CPU and store them
        embedding_batches.append(
            batch_embeddings.cpu()
        )

        total_windows_processed += batch_tensor.shape[0]

        # Show the first few batches
        if batch_number <= 3:
            print(
                f"Batch {batch_number}: "
                f"input {batch_tensor.shape} → "
                f"embeddings {batch_embeddings.shape}"
            )

# Combine all window embeddings into one genome-level collection
all_window_embeddings = torch.cat(
    embedding_batches,
    dim=0
)

print("\nTotal windows processed:", total_windows_processed)
print("Expected windows:", len(representative_windows))
print(
    "All window embeddings shape:",
    all_window_embeddings.shape
)

if total_windows_processed == len(representative_windows):
    print("\n✓ Complete genome processed exactly once.")
else:
    print("\n✗ Window count mismatch.")

Batch 1: input torch.Size([128, 500, 4]) → embeddings torch.Size([128, 128])
Batch 2: input torch.Size([128, 500, 4]) → embeddings torch.Size([128, 128])
Batch 3: input torch.Size([128, 500, 4]) → embeddings torch.Size([128, 128])

Total windows processed: 10885
Expected windows: 10885
All window embeddings shape: torch.Size([10885, 128])

✓ Complete genome processed exactly once.


In [56]:
# Apply Attention-MIL to all window embeddings from the genome

with torch.no_grad():

    genome_embedding, attention_weights, prediction_logit = model.mil(
        all_window_embeddings
    )

print("Window embeddings shape:", all_window_embeddings.shape)
print("Genome embedding shape:", genome_embedding.shape)
print("Attention weights shape:", attention_weights.shape)
print("Prediction logit:", prediction_logit)
print("Attention weights sum:", attention_weights.sum().item())

Window embeddings shape: torch.Size([10885, 128])
Genome embedding shape: torch.Size([128])
Attention weights shape: torch.Size([10885])
Prediction logit: tensor(-0.1614)
Attention weights sum: 1.0000009536743164


In [57]:
# Validate the attention distribution

attention_np = attention_weights.detach().cpu().numpy()

print("Number of attention weights:", len(attention_np))
print("Minimum attention weight:", attention_np.min())
print("Maximum attention weight:", attention_np.max())
print("Mean attention weight:", attention_np.mean())
print("Sum of attention weights:", attention_np.sum())

# Check that all weights are non-negative
non_negative = np.all(attention_np >= 0)

# Check that they sum approximately to 1
sums_to_one = np.isclose(attention_np.sum(), 1.0)

print("\nNon-negative weights:", non_negative)
print("Weights sum to 1:", sums_to_one)

if non_negative and sums_to_one:
    print("\n✓ Attention distribution validation passed.")
else:
    print("\n✗ Attention distribution validation failed.")

Number of attention weights: 10885
Minimum attention weight: 8.944176e-05
Maximum attention weight: 9.4278585e-05
Mean attention weight: 9.186965e-05
Sum of attention weights: 1.0000012

Non-negative weights: True
Weights sum to 1: True

✓ Attention distribution validation passed.


## Step 30 — Define Genome-Level Training Data Stream

During training, each genome is treated as one multiple-instance learning
(MIL) bag.

Windows are generated and one-hot encoded in batches rather than storing
the complete window dataset in memory.

For each genome:

1. Load the corresponding FASTA file.
2. Parse the genome into contigs.
3. Generate windows independently within each contig.
4. One-hot encode windows in batches.
5. Return the batches together with the genome-level label.

The genome label is applied only after all window embeddings have been
aggregated by Attention-MIL.

In [58]:
def generate_genome_training_batches(
    genome_id,
    batch_size=WINDOW_BATCH_SIZE
):
    """
    Stream one-hot encoded window batches from a single genome.

    The genome remains one MIL bag.
    Windows are generated contig-by-contig and processed in batches.

    Yields:
        batch_tensor:
            NumPy array of shape
            (batch_size, 500, 4)

        label:
            Genome-level binary phenotype label.
    """

    # Locate FASTA
    fasta_path = genome_dir / f"{genome_id}.fna"

    if not fasta_path.exists():
        raise FileNotFoundError(
            f"FASTA file not found for Genome ID: {genome_id}"
        )

    # Get genome-level label
    genome_row = model_df[
        model_df["Genome ID"] == genome_id
    ]

    if len(genome_row) != 1:
        raise ValueError(
            f"Expected exactly one phenotype record for {genome_id}, "
            f"found {len(genome_row)}"
        )

    label = int(genome_row.iloc[0]["Label"])

    # Parse genome
    contigs = parse_fasta(fasta_path)

    # Process contigs independently
    batch_windows = []

    for contig_id, sequence in contigs.items():

        # Generate windows from this contig
        contig_windows = generate_windows(
            sequence,
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        for window in contig_windows:

            batch_windows.append(
                window["Sequence"]
            )

            # Yield a full batch
            if len(batch_windows) == batch_size:

                batch_encoded = np.stack([
                    one_hot_encode(seq)
                    for seq in batch_windows
                ])

                yield batch_encoded, label

                batch_windows = []

    # Yield final partial batch
    if batch_windows:

        batch_encoded = np.stack([
            one_hot_encode(seq)
            for seq in batch_windows
        ])

        yield batch_encoded, label

In [59]:
# Test genome-level training batches

training_batch_count = 0
training_window_count = 0
training_labels = []

for batch, label in generate_genome_training_batches(
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
):

    training_batch_count += 1
    training_window_count += batch.shape[0]
    training_labels.append(label)

    if training_batch_count <= 3:
        print(
            f"Batch {training_batch_count}:",
            batch.shape,
            "| Label:",
            label
        )

print("\nTotal batches:", training_batch_count)
print("Total windows:", training_window_count)
print("Expected windows:", len(test_genome_windows))
print("Unique labels:", set(training_labels))

Batch 1: (128, 500, 4) | Label: 0
Batch 2: (128, 500, 4) | Label: 0
Batch 3: (128, 500, 4) | Label: 0

Total batches: 86
Total windows: 10885
Expected windows: 10885
Unique labels: {0}


## Step 31 — Define Genome-Level Training Forward Pass

A complete genome may contain thousands of windows, so windows are passed
through the CNN in smaller batches.

The resulting window embeddings are retained for the current genome and
then passed together to Attention-MIL.

The loss is therefore calculated from a single genome-level prediction,
preserving the MIL formulation.

In [62]:
def forward_genome(
    model,
    genome_id,
    batch_size=WINDOW_BATCH_SIZE,
    device="cpu"
):
    """
    Perform a differentiable forward pass for one genome.

    Returns:
        genome_embedding
        attention_weights
        prediction_logit
        label
    """

    embedding_batches = []
    label = None

    for batch, batch_label in generate_genome_training_batches(
        genome_id,
        batch_size=batch_size
    ):

        # Convert NumPy → PyTorch
        batch_tensor = torch.tensor(
            batch,
            dtype=torch.float32,
            device=device
        )

        # CNN forward pass
        batch_embeddings = model.cnn(
            batch_tensor
        )

        embedding_batches.append(
            batch_embeddings
        )

        label = batch_label

    # Combine all windows belonging to this genome
    all_embeddings = torch.cat(
        embedding_batches,
        dim=0
    )

    # Attention-MIL aggregation
    (
        genome_embedding,
        attention_weights,
        prediction_logit
    ) = model.mil(
        all_embeddings
    )

    return (
        genome_embedding,
        attention_weights,
        prediction_logit,
        label
    )

In [63]:
# Test differentiable genome-level forward pass

model.train()

(
    genome_embedding,
    attention_weights,
    prediction_logit,
    label
) = forward_genome(
    model,
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
)

print("Genome embedding:", genome_embedding.shape)
print("Attention weights:", attention_weights.shape)
print("Prediction logit:", prediction_logit.shape)
print("Genome label:", label)

print("\nNumber of windows:", len(attention_weights))
print("Attention sum:", attention_weights.sum().item())

Genome embedding: torch.Size([128])
Attention weights: torch.Size([10885])
Prediction logit: torch.Size([])
Genome label: 0

Number of windows: 10885
Attention sum: 1.0000009536743164


## Step 32 — Test Genome-Level Loss and Gradient Flow

The model predicts antibiotic resistance at the genome level.

Because the task is binary classification (Resistant vs Susceptible),
BCEWithLogitsLoss is used directly on the genome-level prediction logit.

This step verifies that gradients can flow through the complete model:

Genome windows → CNN → Attention MIL → classifier → loss.

In [64]:
# Step 32 — Test genome-level loss and gradient flow

model.train()

# Clear any existing gradients
model.zero_grad()

# Forward pass through one complete genome
(
    genome_embedding,
    attention_weights,
    prediction_logit,
    label
) = forward_genome(
    model,
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
)

# Convert genome label to PyTorch tensor
label_tensor = torch.tensor(
    float(label),
    dtype=torch.float32,
    device=prediction_logit.device
)

# Binary classification loss
criterion = torch.nn.BCEWithLogitsLoss()

loss = criterion(
    prediction_logit,
    label_tensor
)

print("Prediction logit:", prediction_logit.item())
print("Genome label:", label)
print("Loss:", loss.item())

# Backpropagation
loss.backward()

print("\nGradient flow verified.")

Prediction logit: -0.1613689512014389
Genome label: 0
Loss: 0.6157141327857971

Gradient flow verified.


In [66]:
# Check that trainable model parameters received gradients

gradient_parameters = 0
gradient_norm = 0.0

for name, parameter in model.named_parameters():

    if parameter.requires_grad and parameter.grad is not None:

        gradient_parameters += 1

        gradient_norm += (
            parameter.grad.detach().norm().item()
        ) ** 2

gradient_norm = gradient_norm ** 0.5

print("Parameters with gradients:", gradient_parameters)
print("Total gradient norm:", gradient_norm)

Parameters with gradients: 12
Total gradient norm: 1.606555056970178


## Step 33 — Test One-Genome Optimizer Step

The optimizer updates the CNN, Attention-MIL, and classifier parameters
using the genome-level loss.

A single training step is performed first to verify that model parameters
actually change after backpropagation and optimization.

In [67]:
# Step 33 — Define loss function and optimizer

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [68]:
# Test one complete optimization step

model.train()

# Clear old gradients
optimizer.zero_grad()

# Save one parameter before the update
parameter_name = "cnn.conv1.weight"

parameter_before = (
    model.cnn.conv1.weight.detach()
    .clone()
)

# Forward pass
(
    genome_embedding,
    attention_weights,
    prediction_logit,
    label
) = forward_genome(
    model,
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
)

# Genome-level target
label_tensor = torch.tensor(
    float(label),
    dtype=torch.float32,
    device=prediction_logit.device
)

# Calculate loss
loss = criterion(
    prediction_logit,
    label_tensor
)

# Backpropagation
loss.backward()

# Update parameters
optimizer.step()

# Compare parameter after update
parameter_after = (
    model.cnn.conv1.weight.detach()
)

parameter_change = (
    parameter_after - parameter_before
).norm().item()

print("Genome label:", label)
print("Prediction logit:", prediction_logit.item())
print("Loss:", loss.item())
print("Parameter change:", parameter_change)

Genome label: 0
Prediction logit: -0.1613689512014389
Loss: 0.6157141327857971
Parameter change: 0.06196416914463043


## Step 34 — Fresh Model Initialization and Training Setup

In [69]:
# Step 34 — Initialize the model for the actual experiment

import random
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import accuracy_score, roc_auc_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select available hardware
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Fresh model: discard the optimizer-test update
model = GenomeAMRModel().to(device)

# Training configuration
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Use the existing genome-level splits
train_genome_ids = train_df["Genome ID"].astype(str).tolist()
val_genome_ids = val_df["Genome ID"].astype(str).tolist()
test_genome_ids = test_df["Genome ID"].astype(str).tolist()

# Verify no leakage between partitions
assert set(train_genome_ids).isdisjoint(val_genome_ids)
assert set(train_genome_ids).isdisjoint(test_genome_ids)
assert set(val_genome_ids).isdisjoint(test_genome_ids)

assert set(train_df["Label"].unique()).issubset({0, 1})
assert set(val_df["Label"].unique()).issubset({0, 1})
assert set(test_df["Label"].unique()).issubset({0, 1})

print("Device:", device)
print("Training genomes:", len(train_genome_ids))
print("Validation genomes:", len(val_genome_ids))
print("Test genomes:", len(test_genome_ids))
print("Training labels:", train_df["Label"].value_counts().to_dict())
print("Validation labels:", val_df["Label"].value_counts().to_dict())
print("Fresh model initialized.")

Device: cuda
Training genomes: 796
Validation genomes: 171
Test genomes: 171
Training labels: {0: 464, 1: 332}
Validation labels: {0: 100, 1: 71}
Fresh model initialized.


In [71]:
# Quick GPU and data-path check before full training

print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(model.parameters()).device)
print("Training genome example:", train_genome_ids[0])

example_path = genome_dir / f"{train_genome_ids[0]}.fna"
print("Example FASTA exists:", example_path.exists())
print("Example FASTA path:", example_path)

assert torch.cuda.is_available()
assert next(model.parameters()).is_cuda
assert example_path.exists()

print("GPU and training data checks passed.")

GPU: Tesla T4
Model device: cuda:0
Training genome example: 1280.59769
Example FASTA exists: True
Example FASTA path: /kaggle/input/datasets/prajaktananda1905/amr-data/data/genomes/1280.59769.fna
GPU and training data checks passed.


In [ ]:

# Step 35 — Genome-level CNN + Attention-MIL training
# Includes periodic checkpoints and resume support.

import gc
import time
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, roc_auc_score

# --------------------------------------------------
# 1. Training configuration
# --------------------------------------------------

EPOCHS = 10
CHECKPOINT_EVERY = 10

MODEL_DIR = Path("/kaggle/working/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "best_amr_genome_cnn.pt"
LATEST_MODEL_PATH = MODEL_DIR / "latest_amr_genome_cnn.pt"
TEMP_MODEL_PATH = MODEL_DIR / "latest_amr_genome_cnn.tmp"

best_val_loss = float("inf")
history = []

# --------------------------------------------------
# 2. Resume from the latest checkpoint if available
# --------------------------------------------------

start_epoch = 1
resume_train_index = 0
resume_shuffled_ids = None
resume_train_losses = []

if LATEST_MODEL_PATH.exists():
    checkpoint = torch.load(
        LATEST_MODEL_PATH,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint.get("epoch", 1)
    resume_train_index = checkpoint.get("next_train_index", 0)
    resume_shuffled_ids = checkpoint.get("shuffled_ids")
    resume_train_losses = checkpoint.get("train_losses", [])
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))
    history = checkpoint.get("history", [])

    print("Checkpoint found — resuming training.")
    print("Resume epoch:", start_epoch)
    print("Next training genome index:", resume_train_index + 1)

else:
    print("No previous checkpoint found — starting fresh.")

# --------------------------------------------------
# 3. Save checkpoints safely
# --------------------------------------------------

def save_training_checkpoint(
    epoch,
    next_train_index,
    shuffled_ids,
    train_losses,
    epoch_complete=False
):
    checkpoint_data = {
        "epoch": epoch,
        "next_train_index": next_train_index,
        "shuffled_ids": shuffled_ids,
        "train_losses": train_losses,
        "epoch_complete": epoch_complete,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "seed": SEED
    }

    torch.save(checkpoint_data, TEMP_MODEL_PATH)
    TEMP_MODEL_PATH.replace(LATEST_MODEL_PATH)

# --------------------------------------------------
# 4. Training
# --------------------------------------------------

print("\nDevice:", device)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "CPU"
)
print("Training genomes:", len(train_genome_ids))
print("Validation genomes:", len(val_genome_ids))
print("Test genomes:", len(test_genome_ids))
print("Checkpoint interval:", CHECKPOINT_EVERY, "genomes")
print("\nStarting training...\n")

for epoch in range(start_epoch, EPOCHS + 1):

    epoch_start = time.time()
    model.train()

    # Resume the unfinished epoch, or create a new shuffled order.
    if (
        epoch == start_epoch
        and resume_shuffled_ids is not None
        and resume_train_index > 0
    ):
        shuffled_ids = resume_shuffled_ids
        train_losses = resume_train_losses
        first_index = resume_train_index
    else:
        shuffled_ids = train_genome_ids.copy()
        random.shuffle(shuffled_ids)
        train_losses = []
        first_index = 0

    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    # -------------------------
    # Training genomes
    # -------------------------

    for index in range(first_index, len(shuffled_ids)):

        genome_id = shuffled_ids[index]
        genome_start = time.time()

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"Genome {index + 1}/{len(shuffled_ids)} | "
            f"ID: {genome_id}",
            flush=True
        )

        optimizer.zero_grad(set_to_none=True)

        genome_embedding, attention_weights, logit, label = forward_genome(
            model,
            genome_id,
            batch_size=WINDOW_BATCH_SIZE,
            device=device
        )

        target = torch.tensor(
            float(label),
            dtype=torch.float32,
            device=device
        )

        loss = criterion(logit.reshape(()), target)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        genome_elapsed = (time.time() - genome_start) / 60

        del genome_embedding, attention_weights, logit, target, loss

        if (index + 1) % CHECKPOINT_EVERY == 0:
            save_training_checkpoint(
                epoch=epoch,
                next_train_index=index + 1,
                shuffled_ids=shuffled_ids,
                train_losses=train_losses
            )

            print(
                f"Checkpoint saved | "
                f"Completed {index + 1}/{len(shuffled_ids)} genomes | "
                f"Last genome: {genome_elapsed:.2f} min",
                flush=True
            )

        gc.collect()

    train_loss = float(np.mean(train_losses))

    # -------------------------
    # Validation
    # -------------------------

    model.eval()

    val_losses = []
    val_labels = []
    val_probs = []

    with torch.no_grad():
        for i, genome_id in enumerate(val_genome_ids, start=1):

            print(
                f"Validation {i}/{len(val_genome_ids)} | "
                f"ID: {genome_id}",
                flush=True
            )

            genome_embedding, attention_weights, logit, label = forward_genome(
                model,
                genome_id,
                batch_size=WINDOW_BATCH_SIZE,
                device=device
            )

            target = torch.tensor(
                float(label),
                dtype=torch.float32,
                device=device
            )

            val_losses.append(
                criterion(logit.reshape(()), target).item()
            )
            val_labels.append(int(label))
            val_probs.append(torch.sigmoid(logit).item())

            del genome_embedding, attention_weights, logit, target

    val_loss = float(np.mean(val_losses))
    val_predictions = [int(p >= 0.5) for p in val_probs]

    val_accuracy = accuracy_score(val_labels, val_predictions)

    val_auc = (
        roc_auc_score(val_labels, val_probs)
        if len(set(val_labels)) == 2
        else float("nan")
    )

    # -------------------------
    # Record metrics
    # -------------------------

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_auc": val_auc
    }

    history.append(epoch_result)

    elapsed = (time.time() - epoch_start) / 60

    print(f"\nEpoch {epoch}/{EPOCHS} complete")
    print(f"Train loss:          {train_loss:.4f}")
    print(f"Validation loss:     {val_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.4f}")
    print(f"Validation ROC-AUC:  {val_auc:.4f}")
    print(f"Epoch time:          {elapsed:.1f} minutes")

    # -------------------------
    # Save best model
    # -------------------------

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_auc": val_auc,
            "history": history,
            "seed": SEED
        }

        torch.save(best_checkpoint, BEST_MODEL_PATH)
        print("Best model checkpoint saved.")

    # Save a checkpoint marking this epoch as complete.
    if epoch < EPOCHS:
        save_training_checkpoint(
            epoch=epoch + 1,
            next_train_index=0,
            shuffled_ids=None,
            train_losses=[],
            epoch_complete=True
        )
    else:
        save_training_checkpoint(
            epoch=epoch,
            next_train_index=len(shuffled_ids),
            shuffled_ids=shuffled_ids,
            train_losses=train_losses,
            epoch_complete=True
        )

    print("Latest checkpoint saved.")
    print("-" * 55)

    # Prepare for the next epoch.
    resume_train_index = 0
    resume_shuffled_ids = None
    resume_train_losses = []

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTraining finished.")
print("Best validation loss:", round(best_val_loss, 4))
print("Best model:", BEST_MODEL_PATH)
print("Latest checkpoint:", LATEST_MODEL_PATH)

history_df = pd.DataFrame(history)
display(history_df)


No previous checkpoint found — starting fresh.

Device: cuda
GPU: Tesla T4
Training genomes: 796
Validation genomes: 171
Test genomes: 171
Checkpoint interval: 10 genomes

Starting training...


========== Epoch 1/10 ==========
Epoch 1/10 | Genome 1/796 | ID: 1280.59538
Epoch 1/10 | Genome 2/796 | ID: 1280.59486
Epoch 1/10 | Genome 3/796 | ID: 1280.59308
Epoch 1/10 | Genome 4/796 | ID: 1280.59525
Epoch 1/10 | Genome 5/796 | ID: 1280.9346
Epoch 1/10 | Genome 6/796 | ID: 1280.51746
Epoch 1/10 | Genome 7/796 | ID: 1280.59664
Epoch 1/10 | Genome 8/796 | ID: 1280.9239
Epoch 1/10 | Genome 9/796 | ID: 1280.59597
Epoch 1/10 | Genome 10/796 | ID: 1280.59853
Checkpoint saved | Completed 10/796 genomes | Last genome: 0.05 min
Epoch 1/10 | Genome 11/796 | ID: 1280.59423
Epoch 1/10 | Genome 12/796 | ID: 1280.51766
Epoch 1/10 | Genome 13/796 | ID: 1280.9375
Epoch 1/10 | Genome 14/796 | ID: 1280.15911
Epoch 1/10 | Genome 15/796 | ID: 1280.26069
Epoch 1/10 | Genome 16/796 | ID: 1280.59654
Epoch 1/10 |

In [74]:

import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/working/models/latest_amr_genome_cnn.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Saved epoch:", checkpoint.get("epoch"))
print("Next genome index:", checkpoint.get("next_train_index"))
print("Epoch complete:", checkpoint.get("epoch_complete"))
print("Saved training losses:", len(checkpoint.get("train_losses", [])))
print("History entries:", len(checkpoint.get("history", [])))


Saved epoch: 10
Next genome index: 590
Epoch complete: False
Saved training losses: 590
History entries: 9


In [75]:

print("Saved epoch:", checkpoint["epoch"])
print("Next index:", checkpoint["next_train_index"])
print("Shuffled genome IDs:", len(checkpoint["shuffled_ids"]))
print("Training losses:", len(checkpoint["train_losses"]))
print("Epoch complete:", checkpoint["epoch_complete"])
print("History epochs:", [item["epoch"] for item in checkpoint["history"]])


Saved epoch: 10
Next index: 590
Shuffled genome IDs: 796
Training losses: 590
Epoch complete: False
History epochs: [1, 2, 3, 4, 5, 6, 7, 8, 9]


In [76]:

required = [
    "model",
    "optimizer",
    "forward_genome",
    "train_genome_ids",
    "val_genome_ids",
    "criterion",
    "WINDOW_BATCH_SIZE",
    "device",
    "SEED"
]

for name in required:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")


model: available
optimizer: available
forward_genome: available
train_genome_ids: available
val_genome_ids: available
criterion: available
WINDOW_BATCH_SIZE: available
device: available
SEED: available


In [77]:

print("Optimizer state:", "optimizer_state_dict" in checkpoint)
print("Model state:", "model_state_dict" in checkpoint)
print("Saved genome order:", "shuffled_ids" in checkpoint)
print("Saved next index:", checkpoint.get("next_train_index"))
print("Saved loss count:", len(checkpoint.get("train_losses", [])))


Optimizer state: True
Model state: True
Saved genome order: True
Saved next index: 590
Saved loss count: 590


In [78]:

# Recovery — Resume interrupted Epoch 10

import gc
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, roc_auc_score

CHECKPOINT_PATH = Path("/kaggle/working/models/latest_amr_genome_cnn.pt")
BEST_MODEL_PATH = Path("/kaggle/working/models/best_amr_genome_cnn.pt")
TEMP_PATH = Path("/kaggle/working/models/latest_amr_genome_cnn.tmp")

assert CHECKPOINT_PATH.exists(), "Latest checkpoint is missing."

# Load checkpoint without modifying the original file
ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

assert ckpt["epoch"] == 10, f"Expected epoch 10, found {ckpt['epoch']}."
assert ckpt["epoch_complete"] is False, "This epoch is already marked complete."

model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])

epoch = ckpt["epoch"]
start_index = ckpt["next_train_index"]
shuffled_ids = ckpt["shuffled_ids"]
train_losses = ckpt["train_losses"]
history = ckpt["history"]
best_val_loss = ckpt.get("best_val_loss", float("inf"))

assert len(shuffled_ids) == len(train_genome_ids), "Training genome count mismatch."
assert 0 < start_index < len(shuffled_ids), "Unexpected resume index."
assert len(train_losses) == start_index, "Saved loss count and index disagree."

print(f"Resuming Epoch {epoch} at saved index {start_index}.")
print(f"Already processed: {start_index}/{len(shuffled_ids)}")
print(f"Remaining: {len(shuffled_ids) - start_index}")

def save_recovery_checkpoint(next_index, complete=False):
    data = {
        "epoch": epoch,
        "next_train_index": next_index,
        "shuffled_ids": shuffled_ids,
        "train_losses": train_losses,
        "epoch_complete": complete,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "seed": SEED
    }
    torch.save(data, TEMP_PATH)
    TEMP_PATH.replace(CHECKPOINT_PATH)

# Continue from the saved index
model.train()

for index in range(start_index, len(shuffled_ids)):
    genome_id = shuffled_ids[index]
    genome_start = time.time()

    print(
        f"Epoch 10/10 | Genome {index + 1}/{len(shuffled_ids)} "
        f"| ID: {genome_id}",
        flush=True
    )

    optimizer.zero_grad(set_to_none=True)

    genome_embedding, attention_weights, logit, label = forward_genome(
        model,
        genome_id,
        batch_size=WINDOW_BATCH_SIZE,
        device=device
    )

    target = torch.tensor(
        float(label), dtype=torch.float32, device=device
    )
    loss = criterion(logit.reshape(()), target)
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    del genome_embedding, attention_weights, logit, target, loss

    # Save progress every 10 genomes
    if (index + 1) % 10 == 0 or index == len(shuffled_ids) - 1:
        save_recovery_checkpoint(index + 1)
        print(
            f"Checkpoint saved: {index + 1}/{len(shuffled_ids)} "
            f"| Last genome: {(time.time() - genome_start) / 60:.2f} min",
            flush=True
        )

    gc.collect()

# Compute training loss for the complete epoch
train_loss = float(np.mean(train_losses))

# Run validation
model.eval()
val_losses, val_labels, val_probs = [], [], []

with torch.no_grad():
    for i, genome_id in enumerate(val_genome_ids, start=1):
        print(f"Validation {i}/{len(val_genome_ids)}", flush=True)

        genome_embedding, attention_weights, logit, label = forward_genome(
            model,
            genome_id,
            batch_size=WINDOW_BATCH_SIZE,
            device=device
        )

        target = torch.tensor(
            float(label), dtype=torch.float32, device=device
        )

        val_losses.append(
            criterion(logit.reshape(()), target).item()
        )
        val_labels.append(int(label))
        val_probs.append(torch.sigmoid(logit).item())

        del genome_embedding, attention_weights, logit, target

val_loss = float(np.mean(val_losses))
val_predictions = [int(p >= 0.5) for p in val_probs]
val_accuracy = accuracy_score(val_labels, val_predictions)
val_auc = (
    roc_auc_score(val_labels, val_probs)
    if len(set(val_labels)) == 2 else float("nan")
)

result = {
    "epoch": 10,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_accuracy": val_accuracy,
    "val_auc": val_auc
}
history.append(result)

print("\nEpoch 10/10 complete")
print(f"Training loss:       {train_loss:.4f}")
print(f"Validation loss:     {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation ROC-AUC:  {val_auc:.4f}")

# Update best model only if validation loss improves
if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save({
        "epoch": 10,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_auc": val_auc,
        "history": history,
        "seed": SEED
    }, BEST_MODEL_PATH)
    print("Best model checkpoint updated.")

# Save final checkpoint, marked complete
save_recovery_checkpoint(len(shuffled_ids), complete=True)

print("\nRecovery finished.")
print("Best model:", BEST_MODEL_PATH)
print("Latest checkpoint:", CHECKPOINT_PATH)
display(pd.DataFrame(history))


Resuming Epoch 10 at saved index 590.
Already processed: 590/796
Remaining: 206
Epoch 10/10 | Genome 591/796 | ID: 1280.26038
Epoch 10/10 | Genome 592/796 | ID: 1280.59616
Epoch 10/10 | Genome 593/796 | ID: 1280.26045
Epoch 10/10 | Genome 594/796 | ID: 1280.51815
Epoch 10/10 | Genome 595/796 | ID: 1280.59308
Epoch 10/10 | Genome 596/796 | ID: 1280.9274
Epoch 10/10 | Genome 597/796 | ID: 1280.59258
Epoch 10/10 | Genome 598/796 | ID: 1280.9297
Epoch 10/10 | Genome 599/796 | ID: 1280.9268
Epoch 10/10 | Genome 600/796 | ID: 1280.59287
Checkpoint saved: 600/796 | Last genome: 0.05 min
Epoch 10/10 | Genome 601/796 | ID: 1280.59682
Epoch 10/10 | Genome 602/796 | ID: 1280.59732
Epoch 10/10 | Genome 603/796 | ID: 1280.25996
Epoch 10/10 | Genome 604/796 | ID: 1280.51823
Epoch 10/10 | Genome 605/796 | ID: 1280.59254
Epoch 10/10 | Genome 606/796 | ID: 1280.9403
Epoch 10/10 | Genome 607/796 | ID: 1280.59536
Epoch 10/10 | Genome 608/796 | ID: 1280.51814
Epoch 10/10 | Genome 609/796 | ID: 1280.26049


,epoch,train_loss,val_loss,val_accuracy,val_auc
0,1,0.685776,0.679562,0.584795,0.682887
1,2,0.682257,0.678730,0.584795,0.788521
2,3,0.682012,0.679635,0.584795,0.802606
3,4,0.691991,0.678764,0.584795,0.662676
4,5,0.681978,0.678748,0.584795,0.665141
5,6,0.680994,0.678717,0.584795,0.671197
6,7,0.681034,0.678920,0.584795,0.687183
7,8,0.680876,0.678787,0.584795,0.707958
8,9,0.680851,0.678913,0.584795,0.721549
9,10,0.680338,0.679441,0.584795,0.727113


In [79]:
from pathlib import Path
import torch

best_path = Path("/kaggle/working/models/best_amr_genome_cnn.pt")
latest_path = Path("/kaggle/working/models/latest_amr_genome_cnn.pt")

print("Best checkpoint exists:", best_path.exists())
print("Latest checkpoint exists:", latest_path.exists())
print("Model in memory:", "model" in globals())
print("Test IDs in memory:", "test_genome_ids" in globals())
print("Evaluation function available:", "forward_genome" in globals())

Best checkpoint exists: True
Latest checkpoint exists: True
Model in memory: True
Test IDs in memory: True
Evaluation function available: True


In [80]:
import torch
from pathlib import Path

BEST_MODEL_PATH = Path("/kaggle/working/models/best_amr_genome_cnn.pt")

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best checkpoint loaded successfully!")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation loss:", round(checkpoint["val_loss"], 4))
print("Validation ROC-AUC:", round(checkpoint["val_auc"], 4))
print("Evaluation device:", device)

Best checkpoint loaded successfully!
Checkpoint epoch: 6
Validation loss: 0.6787
Validation ROC-AUC: 0.6712
Evaluation device: cuda


In [81]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

model.eval()

test_labels = []
test_probs = []

print(f"Evaluating {len(test_genome_ids)} test genomes...")

with torch.no_grad():
    for i, genome_id in enumerate(test_genome_ids, start=1):

        genome_embedding, attention_weights, logit, label = forward_genome(
            model,
            genome_id,
            batch_size=WINDOW_BATCH_SIZE,
            device=device
        )

        probability = torch.sigmoid(logit).item()

        test_labels.append(int(label))
        test_probs.append(probability)

        del genome_embedding, attention_weights, logit

        if i % 10 == 0 or i == len(test_genome_ids):
            print(f"Evaluated {i}/{len(test_genome_ids)} genomes")

y_true = np.array(test_labels)
y_probs = np.array(test_probs)
y_pred = (y_probs >= 0.5).astype(int)

print("\nTEST SET RESULTS")
print("=" * 45)
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_probs), 4))
print("PR-AUC (average precision):", round(average_precision_score(y_true, y_probs), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Susceptible (0)", "Resistant (1)"],
    zero_division=0
))

Evaluating 171 test genomes...
Evaluated 10/171 genomes
Evaluated 20/171 genomes
Evaluated 30/171 genomes
Evaluated 40/171 genomes
Evaluated 50/171 genomes
Evaluated 60/171 genomes
Evaluated 70/171 genomes
Evaluated 80/171 genomes
Evaluated 90/171 genomes
Evaluated 100/171 genomes
Evaluated 110/171 genomes
Evaluated 120/171 genomes
Evaluated 130/171 genomes
Evaluated 140/171 genomes
Evaluated 150/171 genomes
Evaluated 160/171 genomes
Evaluated 170/171 genomes
Evaluated 171/171 genomes

TEST SET RESULTS
Accuracy: 0.5848
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.6246
PR-AUC (average precision): 0.5195

Confusion matrix:
[[100   0]
 [ 71   0]]

Classification report:
                 precision    recall  f1-score   support

Susceptible (0)       0.58      1.00      0.74       100
  Resistant (1)       0.00      0.00      0.00        71

       accuracy                           0.58       171
      macro avg       0.29      0.50      0.37       171
   weighted avg       0.34    

In [82]:
import numpy as np
import pandas as pd

# Inspect the variables produced by the test evaluation
required = ["y_true", "y_probs"]

for name in required:
    print(f"{name} available:", name in globals())

if all(name in globals() for name in required):
    y_true_arr = np.asarray(y_true).astype(int).ravel()
    y_probs_arr = np.asarray(y_probs).astype(float).ravel()

    print("\nNumber of test genomes:", len(y_true_arr))
    print("Actual labels:", np.unique(y_true_arr, return_counts=True))
    print("\nProbability summary:")
    print(pd.Series(y_probs_arr).describe())

    print("\nProbabilities by actual class:")
    results = pd.DataFrame({
        "actual_label": y_true_arr,
        "resistance_probability": y_probs_arr
    })

    print(
        results.groupby("actual_label")["resistance_probability"]
        .agg(["count", "mean", "std", "min", "median", "max"])
        .rename(index={0: "Susceptible", 1: "Resistant"})
    )

    print("\nPredictions at different thresholds:")
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
        predicted = (y_probs_arr >= threshold).astype(int)
        print(
            f"Threshold {threshold:.1f}: "
            f"{predicted.sum()} genomes predicted resistant"
        )
else:
    print(
        "\nThose variable names are unavailable. "
        "We'll inspect the evaluation cell to identify the correct names."
    )

y_true available: True
y_probs available: True

Number of test genomes: 171
Actual labels: (array([0, 1]), array([100,  71]))

Probability summary:
count    171.000000
mean       0.411763
std        0.000007
min        0.411743
25%        0.411757
50%        0.411762
75%        0.411768
max        0.411778
dtype: float64

Probabilities by actual class:
              count      mean       std       min    median       max
actual_label                                                         
Susceptible     100  0.411762  0.000007  0.411743  0.411761  0.411778
Resistant        71  0.411765  0.000007  0.411748  0.411765  0.411777

Predictions at different thresholds:
Threshold 0.3: 171 genomes predicted resistant
Threshold 0.4: 171 genomes predicted resistant
Threshold 0.5: 0 genomes predicted resistant
Threshold 0.6: 0 genomes predicted resistant
Threshold 0.7: 0 genomes predicted resistant


In [83]:
import torch

model.eval()

print("Model architecture:\n")
print(model)

print("\nFinal classifier parameters:")
for name, param in model.named_parameters():
    if any(key in name.lower() for key in ["classifier", "attention", "fc", "output"]):
        print(
            name,
            "| shape:", tuple(param.shape),
            "| mean:", round(param.detach().float().mean().item(), 6),
            "| std:", round(param.detach().float().std().item(), 6)
        )

Model architecture:

GenomeAMRModel(
  (cnn): GenomeCNN(
    (conv1): Conv1d(4, 64, kernel_size=(15,), stride=(1,), padding=(7,))
    (conv2): Conv1d(64, 128, kernel_size=(15,), stride=(1,), padding=(7,))
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (global_pool): AdaptiveMaxPool1d(output_size=1)
    (embedding): Linear(in_features=128, out_features=128, bias=True)
  )
  (mil): AttentionMIL(
    (attention): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=1, bias=True)
    )
    (classifier): Linear(in_features=128, out_features=1, bias=True)
  )
)

Final classifier parameters:
mil.attention.0.weight | shape: (64, 128) | mean: -0.000153 | std: 0.094413
mil.attention.0.bias | shape: (64,) | mean: -0.015613 | std: 0.092581
mil.attention.2.weight | shape: (1, 64) | mean: -0.008695 | std: 0.036751
mil.attention.2.bias | shape: (1,) | m

/tmp/ipykernel_49/154052963.py:15: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  "| std:", round(param.detach().float().std().item(), 6)


In [84]:
import torch

# Check for NaN or infinite model parameters
bad_parameters = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        bad_parameters.append(name)

print("Non-finite parameter tensors:", bad_parameters)
print("All model parameters finite:", len(bad_parameters) == 0)

Non-finite parameter tensors: []
All model parameters finite: True


In [85]:
# Compare CNN embeddings from two different test genomes
model.eval()

genome_a = test_genome_ids[0]
genome_b = test_genome_ids[1]

with torch.no_grad():
    embedding_a, attention_a, logit_a, label_a = forward_genome(
        model, genome_a, batch_size=WINDOW_BATCH_SIZE, device=device
    )

    embedding_b, attention_b, logit_b, label_b = forward_genome(
        model, genome_b, batch_size=WINDOW_BATCH_SIZE, device=device
    )

print("Genome A:", genome_a)
print("Genome B:", genome_b)

print("\nEmbedding A — standard deviation:",
      embedding_a.float().std().item())
print("Embedding B — standard deviation:",
      embedding_b.float().std().item())

print("\nMean absolute difference between embeddings:",
      (embedding_a.float() - embedding_b.float()).abs().mean().item())

print("\nLogit A:", logit_a.item())
print("Logit B:", logit_b.item())

Genome A: 1280.51882
Genome B: 1280.59374

Embedding A — standard deviation: 0.0676109790802002
Embedding B — standard deviation: 0.06760725378990173

Mean absolute difference between embeddings: 1.237905235029757e-05

Logit A: -0.3566935062408447
Logit B: -0.356681227684021


In [87]:
import inspect

for name, obj in globals().copy().items():
    if callable(obj) and (
        "window" in name.lower()
        or "fasta" in name.lower()
        or "genome" in name.lower()
        or "encode" in name.lower()
    ):
        try:
            print(f"{name}{inspect.signature(obj)}")
        except (TypeError, ValueError):
            print(name)

parse_fasta(fasta_path)
generate_windows(sequence, window_size=500, stride=250)
estimate_split_windows(split_df)
process_genome(genome_id)
one_hot_encode(sequence)
generate_window_batches(windows, batch_size=128)
GenomeCNN()
GenomeAMRModel()
generate_genome_training_batches(genome_id, batch_size=128)
forward_genome(model, genome_id, batch_size=128, device='cpu')


In [88]:
import torch

model.eval()

# Select two different genomes from your test set
genome_a = test_genome_ids[0]
genome_b = test_genome_ids[1]

# Parse their FASTA sequences
contigs_a = parse_fasta(genome_dir / f"{genome_a}.fna")
contigs_b = parse_fasta(genome_dir / f"{genome_b}.fna")

print("Genome A:", genome_a)
print("Genome B:", genome_b)
print("Contigs A:", len(contigs_a))
print("Contigs B:", len(contigs_b))

Genome A: 1280.51882
Genome B: 1280.59374
Contigs A: 76
Contigs B: 43


In [89]:
import torch
import numpy as np

model.eval()

# Get the first contig sequence from each genome
seq_a = next(iter(contigs_a.values())) if isinstance(contigs_a, dict) else contigs_a[0][1]
seq_b = next(iter(contigs_b.values())) if isinstance(contigs_b, dict) else contigs_b[0][1]

# Generate 500 bp windows
windows_a = generate_windows(seq_a, window_size=500, stride=250)
windows_b = generate_windows(seq_b, window_size=500, stride=250)

print("Windows from Genome A's first contig:", len(windows_a))
print("Windows from Genome B's first contig:", len(windows_b))

Windows from Genome A's first contig: 336
Windows from Genome B's first contig: 165


In [92]:
print("Type of contigs_a:", type(contigs_a))
print("Type of first item in contigs_a:", type(next(iter(contigs_a.values()))))

first_item = next(iter(contigs_a.values()))

if isinstance(first_item, dict):
    print("Keys inside first contig item:", list(first_item.keys())[:10])
elif isinstance(first_item, str):
    print("First DNA sequence preview:", first_item[:80])
else:
    print("First item preview:", repr(first_item)[:200])

Type of contigs_a: <class 'dict'>
Type of first item in contigs_a: <class 'str'>
First DNA sequence preview: GAAAATAATCTCTTCAATTTATTTTTATATGAATCCTGTGACTCAATGATTGTAATATCTAAAGATTTCAGTTCATCATA


In [95]:
# Compare CNN features from two different genomes
import numpy as np
import torch

def get_cnn_features(contigs):
    # Select one DNA sequence from the genome
    sequences = list(contigs.values())
    selected = [sequences[0], sequences[min(1, len(sequences) - 1)]]

    # Encode the selected sequences
   # Use equal-length 500 bp windows from each contig
    selected_windows = [seq[:500] for seq in selected if len(seq) >= 500]
    batch = np.stack([one_hot_encode(seq) for seq in selected_windows])
    x = torch.tensor(batch, dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        features = model.cnn(x)

    return features.detach().cpu().numpy()

features_a = get_cnn_features(contigs_a)
features_b = get_cnn_features(contigs_b)

print("Genome A CNN features:", features_a.shape)
print("Genome B CNN features:", features_b.shape)
print("Mean absolute CNN feature difference:",
      float(np.abs(features_a.mean(axis=0) - features_b.mean(axis=0)).mean()))

Genome A CNN features: (2, 128)
Genome B CNN features: (2, 128)
Mean absolute CNN feature difference: 0.0007439024047926068


In [96]:
# Compare the CNN features before and after training?
# First, inspect the trained CNN's activation scale.

import torch

model.eval()

with torch.no_grad():
    print("CNN parameter summary:")
    for name, param in model.cnn.named_parameters():
        print(
            f"{name}: mean={param.mean().item():.6f}, "
            f"std={param.std().item():.6f}"
        )

print("\nSampled contig lengths:")
print("Genome A:", [len(seq) for seq in list(contigs_a.values())[:5]])
print("Genome B:", [len(seq) for seq in list(contigs_b.values())[:5]])

CNN parameter summary:
conv1.weight: mean=-0.053365, std=0.072473
conv1.bias: mean=-0.057694, std=0.073536
conv2.weight: mean=-0.009814, std=0.019957
conv2.bias: mean=-0.027448, std=0.017554
embedding.weight: mean=-0.000051, std=0.052762
embedding.bias: mean=0.001839, std=0.065529

Sampled contig lengths:
Genome A: [84417, 71050, 8449, 1918, 38366]
Genome B: [41660, 27242, 102545, 38842, 17912]


In [97]:
import pandas as pd
from pathlib import Path

# Inspect the recorded training history, if it is still in memory
if "history" in globals() and history:
    history_df = pd.DataFrame(history)
    display(history_df)
else:
    print("Training history is not available in memory.")

# Inspect checkpoint metadata
checkpoint = torch.load(
    Path("models") / "best_amr_genome_cnn.pt",
    map_location=device,
    weights_only=False
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Training loss:", checkpoint.get("train_loss"))
print("Validation loss:", checkpoint.get("val_loss"))
print("Validation accuracy:", checkpoint.get("val_accuracy"))
print("Validation ROC-AUC:", checkpoint.get("val_auc"))

,epoch,train_loss,val_loss,val_accuracy,val_auc
0,1,0.685776,0.679562,0.584795,0.682887
1,2,0.682257,0.678730,0.584795,0.788521
2,3,0.682012,0.679635,0.584795,0.802606
3,4,0.691991,0.678764,0.584795,0.662676
4,5,0.681978,0.678748,0.584795,0.665141
5,6,0.680994,0.678717,0.584795,0.671197
6,7,0.681034,0.678920,0.584795,0.687183
7,8,0.680876,0.678787,0.584795,0.707958
8,9,0.680851,0.678913,0.584795,0.721549
9,10,0.680338,0.679441,0.584795,0.727113


Best checkpoint epoch: 6
Training loss: 0.6809944755692577
Validation loss: 0.678717163222575
Validation accuracy: 0.5847953216374269
Validation ROC-AUC: 0.6711971830985916


In [98]:
print("Best epoch:", checkpoint["epoch"])
print("Best validation ROC-AUC:", checkpoint["val_auc"])

# Check how many distinct predictions the model makes
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nModel restored from best checkpoint.")
print("CNN device:", next(model.cnn.parameters()).device)
print("MIL device:", next(model.mil.parameters()).device)

Best epoch: 6
Best validation ROC-AUC: 0.6711971830985916

Model restored from best checkpoint.
CNN device: cuda:0
MIL device: cuda:0
